# Monte Carlo — refined strategy set

Bootstrap risk analysis of the **ablation-refined** strategy configurations
(2026-07-18 full-matrix ablations, `ablate.py` + `ablate_vbt.py`):

| Sleeve | Refined rules | Why |
|---|---|---|
| residualmomentum | RA-RES only | all 3 confirmation legs are drag |
| macd | CROSS + STAGE=sub0 | EMA50 / RSI dead weight |
| bollinger | full 4-leg, **choppy regime only** | its routing slot; all-regime view misleads |
| vwap | AVWAP + VOL + RSI-RESET | EMA-STACK mildly dead |
| pead | event gate only | no-rules baseline beats every combo |
| consolidation | — retired | COMP leg toxic |

Each sleeve's trades come from the vectorized VBT backtest
(`rhengine/mc_refined.py`, same fills/exits/costs as `ablate_vbt.py`, 5 bps
per side, 730d full-market Massive cache). Per-trade **R-multiples**
(trade return ÷ stop-distance fraction at entry) are resampled with
replacement into thousands of equity paths under fixed-fractional sizing —
so one lucky trade *ordering* can't hide tail risk.

**Run top-to-bottom** (VS Code: open and *Run All*). If the trades CSV is
missing it will be harvested first (several minutes, full market).

In [ ]:
from pathlib import Path
import subprocess, sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd() if (Path.cwd() / "config.json").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from rhengine.mc_refined import run_mc_paths, REFINED_SET

# ---- parameters --------------------------------------------------------
TRADES_CSV = ROOT / "tests/results/refined_trades.csv"
REFRESH    = False      # True -> re-run the harvest even if the CSV exists
PATHS      = 20_000     # bootstrap paths (findings spec: >= 20k)
N_TRADES   = 250        # trades per path (~a year of signals per sleeve)
RISK_PCT   = 1.0        # equity % risked per trade (1R)
RUIN_DD    = 50.0       # drawdown % treated as "ruin"
SEED       = 1337

# ---- palette (dataviz reference, light mode; fixed slot order) ---------
HUES = {"residualmomentum": "#2a78d6", "macd": "#008300",
        "bollinger": "#e87ba4", "vwap": "#eda100", "pead": "#1baf7a",
        "POOLED": "#0b0b0b"}
INK, INK2, MUTED = "#0b0b0b", "#52514e", "#898781"
GRID, BASELINE, SURFACE = "#e1e0d9", "#c3c2b7", "#fcfcfb"

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "savefig.facecolor": SURFACE, "figure.dpi": 110,
    "font.family": "Segoe UI", "font.size": 10,
    "text.color": INK, "axes.labelcolor": INK2,
    "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.edgecolor": BASELINE, "axes.linewidth": 0.8,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlecolor": INK, "axes.titlesize": 11,
    "legend.frameon": False,
})

## 1 · Load the refined-set trades

In [ ]:
if REFRESH or not TRADES_CSV.exists():
    print("Harvesting refined-set trades (full market, ~minutes)...")
    subprocess.run([sys.executable, "-m", "rhengine.mc_refined",
                    "--lookback-days", "730", "--cost-bps", "5",
                    "--csv", str(TRADES_CSV)], cwd=ROOT, check=True)

trades = pd.read_csv(TRADES_CSV, parse_dates=["entry_date", "exit_date"])
ORDER = [s for s in REFINED_SET if s in set(trades["strategy"])]

summary = (trades.groupby("strategy")
           .agg(rules=("rules", "first"), n=("r_multiple", "size"),
                win_rate=("r_multiple", lambda r: (r > 0).mean()),
                avg_R=("r_multiple", "mean"), med_R=("r_multiple", "median"),
                p95_R=("r_multiple", lambda r: r.quantile(0.95)))
           .reindex(ORDER).round(3))
print(f"{len(trades):,} trades, "
      f"{trades.entry_date.min().date()} - {trades.entry_date.max().date()}")
summary

## 2 · Bootstrap the equity paths

For each sleeve (and the pooled set) we resample `N_TRADES` R-multiples with
replacement, `PATHS` times, compounding at `RISK_PCT`% risk per trade. Kept
per run: the percentile **fan** of the equity path, plus the final-equity and
max-drawdown distributions.

In [ ]:
FAN_PCTS = (5, 25, 50, 75, 95)
mc, fans = {}, {}
pools = {s: trades.loc[trades.strategy == s, "r_multiple"].values for s in ORDER}
pools["POOLED"] = trades["r_multiple"].values

for name, rs in pools.items():
    r = run_mc_paths(rs, n_trades=N_TRADES, paths=PATHS,
                     risk_pct=RISK_PCT, seed=SEED, ruin_dd_pct=RUIN_DD)
    fans[name] = np.percentile(r.pop("equity"), FAN_PCTS, axis=0)
    mc[name] = r

risk_tbl = pd.DataFrame({
    name: {"observed trades": r["n_observed"], "avg R": round(r["avg_r"], 3),
           "final ×  p5": round(r["final_pct"][5], 2),
           "final ×  p50": round(r["final_pct"][50], 2),
           "final ×  p95": round(r["final_pct"][95], 2),
           "maxDD %  p50": round(r["maxdd_pct"][50], 1),
           "maxDD %  p95": round(r["maxdd_pct"][95], 1),
           "P(loss) %": round(r["prob_loss"], 1),
           f"P(DD>{RUIN_DD:.0f}%) %": round(r["prob_ruin"], 1)}
    for name, r in mc.items()}).T
risk_tbl

## 3 · Equity fan charts

Each panel: one sleeve's bootstrap paths as percentile bands — outer band
p5–p95, inner p25–p75, heavy line = median — over 250 resampled trades at
the configured risk. The dashed hairline is break-even (1.0×).

In [ ]:
names = ORDER + ["POOLED"]
fig, axes = plt.subplots(2, 3, figsize=(12.5, 6.8), sharex=True)
x = np.arange(1, N_TRADES + 1)
for ax, name in zip(axes.flat, names):
    hue, fan = HUES[name], fans[name]
    ax.fill_between(x, fan[0], fan[4], color=hue, alpha=0.14, lw=0)
    ax.fill_between(x, fan[1], fan[3], color=hue, alpha=0.28, lw=0)
    ax.plot(x, fan[2], color=hue, lw=2)
    ax.axhline(1.0, color=BASELINE, lw=1, ls=(0, (4, 3)))
    for row, p in ((0, "p5"), (2, "p50"), (4, "p95")):
        ax.annotate(f" {p} {fan[row, -1]:.2f}×", (x[-1], fan[row, -1]),
                    fontsize=8, color=INK2 if row != 2 else INK, va="center")
    ax.set_title(name if name == "POOLED" else
                 f"{name} · {trades.loc[trades.strategy == name, 'rules'].iat[0]}",
                 loc="left", fontsize=9.5)
    ax.set_xlim(0, N_TRADES * 1.16)
    ax.margins(y=0.06)
for ax in axes[1]:
    ax.set_xlabel("trade #")
for ax in axes[:, 0]:
    ax.set_ylabel("equity multiple")
fig.suptitle(f"Bootstrap equity fans — {PATHS:,} paths × {N_TRADES} trades, "
             f"{RISK_PCT:.1f}% risk/trade", x=0.01, ha="left",
             fontsize=12, fontweight="bold")
fig.tight_layout(rect=(0, 0, 1, 0.95))
plt.show()

## 4 · Median paths, head-to-head

One axis, the median bootstrap path of every sleeve. This is the *typical*
outcome — the fans above show how wide "typical" really is.

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 4.6))
for name in names:
    med = fans[name][2]
    ax.plot(x, med, color=HUES[name], lw=2,
            ls=(0, (5, 2)) if name == "POOLED" else "-")
    ax.annotate(f" {name} {med[-1]:.2f}×", (x[-1], med[-1]),
                color=HUES[name] if name != "vwap" else "#c98500",
                fontsize=9, va="center", fontweight="bold")
ax.axhline(1.0, color=BASELINE, lw=1, ls=(0, (4, 3)))
ax.set_xlim(0, N_TRADES * 1.42)
ax.set_xlabel("trade #"); ax.set_ylabel("median equity multiple")
ax.set_title(f"Median bootstrap path per sleeve ({RISK_PCT:.1f}% risk/trade)",
             loc="left", fontweight="bold")
plt.tight_layout(); plt.show()

## 5 · Outcome distributions

Left: where the paths **end** (final equity multiple). Right: the **worst
drawdown** each path suffered on the way. Vertical hairline: break-even /
the ruin threshold.

In [ ]:
fig, axes = plt.subplots(len(names), 2, figsize=(11, 1.75 * len(names)),
                         sharex="col")
for i, name in enumerate(names):
    hue, r = HUES[name], mc[name]
    axL, axR = axes[i]
    axL.hist(np.clip(r["finals"], 0, np.percentile(r["finals"], 99.5)),
             bins=60, color=hue, alpha=0.85)
    axL.axvline(1.0, color=INK2, lw=1, ls=(0, (4, 3)))
    axL.annotate(f"{name}  ·  P(loss) {r['prob_loss']:.0f}%",
                 (0.99, 0.82), xycoords="axes fraction", ha="right",
                 fontsize=9, color=INK, fontweight="bold")
    axR.hist(r["maxdd"] * 100, bins=60, color=hue, alpha=0.85)
    axR.axvline(RUIN_DD, color="#d03b3b", lw=1, ls=(0, (4, 3)))
    axR.annotate(f"p95 {r['maxdd_pct'][95]:.0f}%  ·  "
                 f"P(DD>{RUIN_DD:.0f}%) {r['prob_ruin']:.1f}%",
                 (0.99, 0.82), xycoords="axes fraction", ha="right",
                 fontsize=9, color=INK2)
    for a in (axL, axR):
        a.set_yticks([])
axes[-1, 0].set_xlabel("final equity multiple")
axes[-1, 1].set_xlabel("max drawdown %")
fig.suptitle("Final equity (left) and max drawdown (right) across paths",
             x=0.01, ha="left", fontsize=12, fontweight="bold")
fig.tight_layout(rect=(0, 0, 1, 0.96))
plt.show()

## 6 · How hot can the risk dial go?

The same pooled bootstrap swept across risk-per-trade settings. Two panels,
one measure each: growth (median / p5 final multiple) and pain (p95 max
drawdown, with the ruin threshold marked). Pick the largest risk whose
**pain** you can actually sit through — the growth panel alone always argues
for more.

In [ ]:
risks = np.arange(0.25, 3.01, 0.25)
sweep = [run_mc_paths(pools["POOLED"], n_trades=N_TRADES, paths=5_000,
                      risk_pct=rp, seed=SEED, ruin_dd_pct=RUIN_DD)
         for rp in risks]
med_f = [s["final_pct"][50] for s in sweep]
p5_f  = [s["final_pct"][5] for s in sweep]
p95dd = [s["maxdd_pct"][95] for s in sweep]

blue = HUES["residualmomentum"]
fig, (axG, axP) = plt.subplots(1, 2, figsize=(11, 4.2))
axG.plot(risks, med_f, color=blue, lw=2, marker="o", ms=4)
axG.plot(risks, p5_f, color=blue, lw=2, marker="o", ms=4, ls=(0, (5, 2)))
axG.annotate(" median", (risks[-1], med_f[-1]), color=blue,
             fontsize=9, va="center", fontweight="bold")
axG.annotate(" p5 (bad luck)", (risks[-1], p5_f[-1]), color=INK2,
             fontsize=9, va="center")
axG.axhline(1.0, color=BASELINE, lw=1, ls=(0, (4, 3)))
axG.set_xlim(0.1, 3.75)
axG.set_xlabel("risk per trade %"); axG.set_ylabel("final equity multiple")
axG.set_title("Growth", loc="left", fontweight="bold")

axP.plot(risks, p95dd, color="#d03b3b", lw=2, marker="o", ms=4)
axP.axhline(RUIN_DD, color="#d03b3b", lw=1, ls=(0, (4, 3)))
axP.annotate(f"ruin threshold {RUIN_DD:.0f}%", (0.3, RUIN_DD),
             fontsize=8.5, color="#d03b3b", va="bottom")
axP.set_xlabel("risk per trade %"); axP.set_ylabel("p95 max drawdown %")
axP.set_title("Pain", loc="left", fontweight="bold")
fig.suptitle(f"Pooled refined set — risk sweep ({N_TRADES} trades/path)",
             x=0.01, ha="left", fontsize=12, fontweight="bold")
fig.tight_layout(rect=(0, 0, 1, 0.94))
plt.show()

## Caveats

- **Bootstrap = i.i.d. resampling.** It scrambles trade order, so regime
  clustering (losing streaks in a bad tape) is *understated*; real drawdowns
  can run deeper than the fans suggest.
- Trades come from the **vectorized VBT engine** — same known deviations as
  `ablate_vbt.py` (no chandelier trail, ewm-Wilder smoothing, one position
  per ticker). Rankings and shapes are what to trust, not the third decimal.
- Sleeves are simulated **independently and pooled by concatenation**;
  correlation between sleeves trading the same tape is not modeled. A
  portfolio-level sim (`rhengine/portfolio.py`) is the cross-check.
- Per the findings spec: whole-history numbers only — promotion decisions
  still require the walk-forward / OOS gates, not this notebook.